# BMA — Kaggle Setup and Phase 0 Smoke Test

This notebook is ready to run in Kaggle with **Run All**. It clones or updates the public repository, installs the local `doc_agent` package, verifies the environment and GPU, discovers attached PDF inputs, loads the project configuration, and writes a smoke-test artifact.

Before running, attach the two source PDFs as a **private Kaggle dataset**. The notebook discovers PDFs anywhere under `/kaggle/input`, so the Kaggle dataset slug does not need to be hard-coded.

In [ ]:
# Configuration — normally no edits are required.
from pathlib import Path

REPO_URL = "https://github.com/FAHIM-ISHTIAK/doc-agent-20.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/doc-agent-20")
KAGGLE_INPUT = Path("/kaggle/input")
ARTIFACT_DIR = Path("/kaggle/working/artifacts")

print("Repository:", REPO_URL)
print("Branch:", REPO_BRANCH)
print("Working copy:", REPO_DIR)
print("Kaggle inputs:", KAGGLE_INPUT)


## 1. Inspect the Kaggle runtime

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Current directory:", Path.cwd())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for gpu_index in range(torch.cuda.device_count()):
            print(f"GPU {gpu_index}:", torch.cuda.get_device_name(gpu_index))
    else:
        print("WARNING: No GPU detected. In Kaggle, open Settings -> Accelerator and select a GPU.")
except Exception as exc:
    print("PyTorch inspection failed:", repr(exc))


## 2. Clone or update the repository

The cell is safe to run again. If a clean Git checkout already exists, it updates `main` with a fast-forward pull. It does not delete or overwrite an unrelated directory.

In [ ]:
def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)))
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Choose a different REPO_DIR.")
    print("Existing repository found; updating it.")
    run_command(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_command(["git", "switch", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    print("Cloning repository.")
    run_command(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

commit_hash = run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
status = run_command(["git", "status", "--short"], cwd=REPO_DIR).stdout.strip()
print("Tested commit:", commit_hash)
print("Working tree:", "clean" if not status else status)


## 3. Install the repository package

This smoke test installs the local package without resolving the full ML dependency stack. Phase 2 will finalize and pin the complete Kaggle dependency environment.

In [ ]:
run_command(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"]
)

# Add src explicitly as a robust notebook-session fallback after editable installation.
src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

os.chdir(REPO_DIR)
print("Current directory:", Path.cwd())
print("Source directory on sys.path:", str(src_dir) in sys.path)


## 4. Import `doc_agent` and load its fixed contracts

In [ ]:
import importlib.util

package_spec = importlib.util.find_spec("doc_agent")
print("doc_agent module specification:", package_spec)
if package_spec is None:
    raise ModuleNotFoundError("doc_agent is still unavailable after editable installation")

import doc_agent
from doc_agent import config
from doc_agent.contracts import Chunk, Page, Region

cfg = config.load(REPO_DIR / "configs" / "config.yaml")
task_cfg = config.load_task(REPO_DIR / "configs" / "task.yaml")

print("doc_agent imported from:", Path(doc_agent.__file__).resolve())
print("Configuration loaded successfully:")
print(cfg)
print("Task configuration loaded successfully:")
print(task_cfg)
print("Contracts:", Page.__name__, Region.__name__, Chunk.__name__)
print("IMPORT SMOKE TEST: PASS")


## 5. Discover attached corpus PDFs

The PDFs may live under any private Kaggle input dataset. This cell searches all of `/kaggle/input`.

In [ ]:
if not KAGGLE_INPUT.exists():
    raise FileNotFoundError("/kaggle/input does not exist. This notebook is intended to run in Kaggle.")

pdf_files = sorted(KAGGLE_INPUT.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s) under {KAGGLE_INPUT}:")
for pdf_path in pdf_files:
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"- {pdf_path} ({size_mb:.2f} MB)")

if len(pdf_files) < 2:
    raise FileNotFoundError(
        "Expected the two BMA source PDFs. Attach the private corpus dataset to this Kaggle notebook and run again."
    )

print("CORPUS DISCOVERY: PASS")


## 6. Validate the fixed data contracts

In [ ]:
sample_page = Page(
    id="smoke_page_0001",
    image_path="/kaggle/working/example.png",
    doc_id="smoke_document",
)
sample_region = Region(
    page_id=sample_page.id,
    bbox=(0, 0, 100, 100),
    kind="text",
)
sample_chunk = Chunk(
    id="smoke_chunk_0001",
    doc_id=sample_page.doc_id,
    text="বাংলা ব্যাকরণ",
    page_ids=[sample_page.id],
)

print(sample_page.model_dump())
print(sample_region.model_dump())
print(sample_chunk.model_dump())
assert sample_chunk.text == "বাংলা ব্যাকরণ"
assert sample_chunk.page_ids == [sample_page.id]
print("CONTRACT SMOKE TEST: PASS")


## 7. Verify writable artifact storage

In [ ]:
import json
from datetime import datetime, timezone

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
smoke_artifact = ARTIFACT_DIR / "phase0_smoke_test.json"
smoke_payload = {
    "status": "pass",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL,
    "branch": REPO_BRANCH,
    "commit": commit_hash,
    "python": sys.version,
    "pdf_count": len(pdf_files),
    "pdf_files": [str(path) for path in pdf_files],
}
smoke_artifact.write_text(
    json.dumps(smoke_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(smoke_artifact.read_text(encoding="utf-8"))
print("Artifact written to:", smoke_artifact)
print("WRITABLE STORAGE TEST: PASS")


## Phase 0 result

If every code cell above finishes successfully, the Kaggle portion of Phase 0 is complete:

- the public repository can be cloned or updated;
- the exact tested commit is recorded;
- `doc_agent` imports from the repository's `src/` layout;
- the fixed `Page`, `Region`, and `Chunk` contracts work;
- the private corpus PDFs are visible without hard-coded dataset slugs;
- the selected GPU/runtime is reported;
- output can be written to `/kaggle/working/artifacts`.

Save a Kaggle version with outputs, then download this executed notebook and commit it only if the team chooses this Phase 0/EDA work as an authored contribution. Phase 1 will extend this notebook with the real corpus EDA.